In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.stats import gamma
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix
from scipy.spatial.distance import pdist, squareform
from IPython.display import display
import torch
import torch.nn as nn
import torch.optim as optim
from nilearn.glm.first_level import spm_hrf

# Figure settings
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
sns.set_context("notebook", font_scale=1.2)
plt.rcParams['figure.figsize'] = (10, 6)

# HCP dataset parameters
HCP_DIR = "./hcp_task"
N_SUBJECTS = 100
N_PARCELS = 360
TR = 0.72  # Time resolution in seconds
RUNS = ['LR','RL']
EXPERIMENTS = {
    'MOTOR'      : {'cond':['lf','rf','lh','rh','t','cue']},
    'WM'         : {'cond':['0bk_body','0bk_faces','0bk_places','0bk_tools','2bk_body','2bk_faces','2bk_places','2bk_tools']},
    'EMOTION'    : {'cond':['fear','neut']},
    'GAMBLING'   : {'cond':['loss','win']},
    'LANGUAGE'   : {'cond':['math','story']},
    'RELATIONAL' : {'cond':['match','relation']},
    'SOCIAL'     : {'cond':['ment','rnd']}
}
N_BOOTSTRAP = 2000

# Load region info
regions = np.load(f"{HCP_DIR}/regions.npy").T
region_info = dict(
    name=regions[0].tolist(),
    network=regions[1],
    hemi=['Right']*int(N_PARCELS/2) + ['Left']*int(N_PARCELS/2),
)

def load_single_timeseries(subject, experiment, run, remove_mean=True):
    """Load timeseries data for a single subject and single run.

    Args:
        subject (str):      subject ID to load
        experiment (str):   Name of experiment
        run (int):          (0 or 1)
        remove_mean (bool): If True, subtract the parcel-wise mean (typically the mean BOLD signal is not of interest)

    Returns
        ts (n_parcel x n_timepoint array): Array of BOLD data values

    """
    bold_run  = RUNS[run]
    bold_path = f"{HCP_DIR}/subjects/{subject}/{experiment}/tfMRI_{experiment}_{bold_run}"
    bold_file = "data.npy"
    ts = np.load(f"{bold_path}/{bold_file}")
    if remove_mean:
        ts -= ts.mean(axis=1, keepdims=True)
    return ts

def load_evs(subject, experiment, run):
    """Load EVs (explanatory variables) data for one task experiment.

    Args:
        subject (str): subject ID to load
        experiment (str) : Name of experiment
        run (int): 0 or 1

    Returns
        evs (list of lists): A list of frames associated with each condition

    """
    frames_list = []
    task_key = f'tfMRI_{experiment}_{RUNS[run]}'
    for cond in EXPERIMENTS[experiment]['cond']:
        ev_file  = f"{HCP_DIR}/subjects/{subject}/{experiment}/{task_key}/EVs/{cond}.txt"
        ev_array = np.loadtxt(ev_file, ndmin=2, unpack=True)
        ev       = dict(zip(["onset", "duration", "amplitude"], ev_array))
        start = np.floor(ev["onset"] / TR).astype(int)
        duration = np.ceil(ev["duration"] / TR).astype(int)
        frames = [s + np.arange(0, d) for s, d in zip(start, duration)]
        frames_list.append(frames)
    return frames_list


def condition_frames(run_evs, cond_idx):
    """Return all TR frames for one condition from a run-specific EV list."""
    return np.concatenate(run_evs[cond_idx])

def rest_frames(run_evs, n_tp):
    """Return all TR frames not assigned to any task condition in one run."""
    on_frames = np.unique(np.concatenate([np.concatenate(frames) for frames in run_evs]))
    return np.setdiff1d(np.arange(n_tp), on_frames)

def design_matrix_for_run(run_evs, n_tp, condition_indices=None, hrf=None):
    """Build the GLM design matrix for one subject/run from EV timing.

    The HCP EV files give each condition's onset and duration. Earlier, load_evs()
    converted those timings into frame indices. This function turns those frame indices
    into one 0/1 stimulus vector per condition, where 1 means the condition is ON at
    that fMRI time point and 0 means it is OFF. If an HRF is provided, each 0/1
    stimulus vector is convolved with the HRF to create a delayed, smoothed predicted
    BOLD response.

    Args:
        run_evs (list): Run-specific EV frame lists, one entry per condition.
        n_tp (int): Number of fMRI time points in this run.
        condition_indices (iterable, optional): Condition indices to include. If None,
            include all conditions.
        hrf (np.ndarray, optional): HRF kernel for convolution. If None, use raw 0/1
            stimulus vectors.

    Returns:
        np.ndarray: Design matrix of shape (n_timepoints, n_conditions), with one
        predictor column per condition.
    """
    if condition_indices is None:
        condition_indices = range(len(run_evs))
    X = np.zeros((n_tp, len(condition_indices)))
    for col, cond_idx in enumerate(condition_indices):
        frames = condition_frames(run_evs, cond_idx)
        pred = np.zeros(n_tp)
        pred[frames] = 1.0
        if hrf is not None:
            pred = np.convolve(pred, hrf, mode='full')[:n_tp]
        X[:, col] = pred
    return X

subjects = np.loadtxt(os.path.join(HCP_DIR, "subjects_list.txt"), dtype='str')
my_exp = 'MOTOR'
conditions = EXPERIMENTS[my_exp]['cond']
rh_idx = conditions.index('rh')
lh_idx = conditions.index('lh')

# load every subject's time series and EVs once.
all_ts = []   # all_ts[run][subj_i]  -> (360 x n_tp) array
all_evs = []  # all_evs[run][subj_i] -> list-of-lists of EV frames
for run in range(len(RUNS)):
    ts_run, ev_run = [], []
    for subj in subjects:
        ts_run.append(load_single_timeseries(subject=subj, experiment=my_exp, run=run))
        ev_run.append(load_evs(subject=subj, experiment=my_exp, run=run))
    all_ts.append(ts_run)
    all_evs.append(ev_run)

network_order = [
    'Visual1', 'Visual2', 'Somatomotor', 'Cingulo-Oper', 'Dorsal-atten',
    'Language', 'Frontopariet', 'Auditory', 'Default', 'Posterior-Mu',
    'Ventral-Mult', 'Orbito-Affec'
]
network_label_map = {
    'Visual1': 'Primary Visual',
    'Visual2': 'Secondary Visual',
    'Somatomotor': 'Somatomotor',
    'Cingulo-Oper': 'Cingulo-Opercular',
    'Dorsal-atten': 'Dorsal Attention',
    'Language': 'Language',
    'Frontopariet': 'Frontoparietal',
    'Auditory': 'Auditory',
    'Default': 'Default Mode',
    'Posterior-Mu': 'Posterior Multimodal',
    'Ventral-Mult': 'Ventral Multimodal',
    'Orbito-Affec': 'Orbito-Affective',
}
network_order = [net for net in network_order if net in set(region_info['network'])]
network_labels = [network_label_map[net] for net in network_order]

def fit_condition_betas(ts, run_evs, hrf):
    """Fit a six-condition GLM for one subject/run and return condition x parcel betas."""
    model = LinearRegression()
    X_design = design_matrix_for_run(run_evs, ts.shape[1], hrf=hrf)
    model.fit(X_design, ts.T)
    return model.coef_.T

def compute_subject_condition_betas(hrf):
    """Return subject-level condition betas averaged across LR/RL runs."""
    subject_betas = []
    for subj_i in range(len(subjects)):
        run_betas = []
        for run in range(len(RUNS)):
            run_betas.append(fit_condition_betas(all_ts[run][subj_i], all_evs[run][subj_i], hrf))
        subject_betas.append(np.mean(run_betas, axis=0))
    return np.asarray(subject_betas)  # subjects x conditions x parcels

def benjamini_hochberg(pvals, alpha=0.05):
    """Return FDR-significant tests and Benjamini-Hochberg adjusted p-values."""
    pvals = np.asarray(pvals)
    n_tests = pvals.size
    order = np.argsort(pvals)
    ranked_pvals = pvals[order]
    ranks = np.arange(1, n_tests + 1)

    adjusted_sorted = ranked_pvals * n_tests / ranks
    adjusted_sorted = np.minimum.accumulate(adjusted_sorted[::-1])[::-1]
    adjusted_sorted = np.clip(adjusted_sorted, 0, 1)

    adjusted = np.empty_like(adjusted_sorted)
    adjusted[order] = adjusted_sorted
    significant = adjusted < alpha
    return significant, adjusted


# Q1


In [ ]:
# ON vs. rest visualization using each subject/run's own EV timing.
rest_subject_run = []
for subj_i in range(len(subjects)):
    for run in range(len(RUNS)):
        ts = all_ts[run][subj_i]
        run_evs = all_evs[run][subj_i]
        off = rest_frames(run_evs, ts.shape[1])
        rest_subject_run.append(ts[:, off].mean(axis=1))
activity_off = np.mean(rest_subject_run, axis=0)

# Sort all 360 regions anatomically by Functional Network
# map each region's network to its index in network_order
sort_keys = []
for net in region_info['network']:
    sort_keys.append(network_order.index(net))
        
sorted_indices = np.argsort(sort_keys)
sorted_networks = np.array(region_info['network'])[sorted_indices]

# Find the boundaries and centers for the network labels on the x-axis
network_boundaries = []
network_centers = []
network_names = []
current_net = sorted_networks[0]
start_idx = 0

for i in range(1, len(sorted_networks)):
    if sorted_networks[i] != current_net:
        network_boundaries.append(i)
        network_centers.append(start_idx + (i - start_idx) / 2)
        network_names.append(network_label_map.get(current_net, current_net))
        current_net = sorted_networks[i]
        start_idx = i

# Add the final network
network_boundaries.append(len(sorted_networks))
network_centers.append(start_idx + (len(sorted_networks) - start_idx) / 2)
network_names.append(network_label_map.get(current_net, current_net))

fig, axes = plt.subplots(len(conditions), 1, figsize=(16, 24), sharex=True)

# Generate distinct pastel colors for the network background stripes
network_colors = sns.color_palette("tab20", len(network_names))

for cond_idx, cond in enumerate(conditions):
    on_subject_run = []
    for subj_i in range(len(subjects)):
        for run in range(len(RUNS)):
            ts = all_ts[run][subj_i]
            frames = condition_frames(all_evs[run][subj_i], cond_idx)
            on_subject_run.append(ts[:, frames].mean(axis=1))
    activity_on_cond = np.mean(on_subject_run, axis=0)

    # Add continuous colored stripes for each functional network
    start_bnd = 0
    for i, boundary in enumerate(network_boundaries):
        axes[cond_idx].axvspan(start_bnd, boundary, facecolor=network_colors[i], alpha=0.4, zorder=0)
        start_bnd = boundary

    axes[cond_idx].plot(activity_off[sorted_indices], color='gray', alpha=0.8, linestyle='-', linewidth=1.5, label='OFF', zorder=5)
    axes[cond_idx].plot(activity_on_cond[sorted_indices], color='black', alpha=0.9, linewidth=2, label=f'ON ({cond})', zorder=6)
            
    axes[cond_idx].set_ylabel('BOLD Signal')
    axes[cond_idx].set_ylim(-150, 150)
    axes[cond_idx].set_title(f'{cond.upper()} vs Rest across all 360 brain regions')
    axes[cond_idx].legend(loc='upper right')

plt.xticks(network_centers, network_names, rotation=45, ha='right', fontsize=11)
plt.xlabel('Anatomical Regions (Grouped by Functional Network)', fontsize=14)
plt.xlim(0, 360)
plt.tight_layout()
plt.show()


# Q2

In [ ]:
def hand_and_rest_means_by_subject():
    """
    Compute the subject-level mean BOLD activation for the right hand, left hand, and rest conditions.
    
    For each subject, this function averages the BOLD time series across all frames from the 
    right hand (rh) condition, left hand (lh) condition, and rest (OFF) condition within each run.
    It then averages the 2 runs (RL, LR) to produce a single representative value per parcel per subject.

    Returns:
        rh_means (np.ndarray): Array of shape (n_subjects, n_parcels) containing mean right hand activations.
        lh_means (np.ndarray): Array of shape (n_subjects, n_parcels) containing mean left hand activations.
        rest_means (np.ndarray): Array of shape (n_subjects, n_parcels) containing mean rest activations.
    """
    rh_subject_means = []
    lh_subject_means = []
    rest_subject_means = []

    for subj_i in range(len(subjects)):
        rh_run_means = []
        lh_run_means = []
        rest_run_means = []

        for run in range(len(RUNS)):
            ts      = all_ts[run][subj_i]
            run_evs = all_evs[run][subj_i]

            run_rh_frames   = condition_frames(run_evs, rh_idx)
            run_lh_frames   = condition_frames(run_evs, lh_idx)
            run_rest_frames = rest_frames(run_evs, ts.shape[1])

            rh_run_means.append(ts[:, run_rh_frames].mean(axis=1))
            lh_run_means.append(ts[:, run_lh_frames].mean(axis=1))
            rest_run_means.append(ts[:, run_rest_frames].mean(axis=1))

        rh_subject_means.append(np.mean(rh_run_means, axis=0))
        lh_subject_means.append(np.mean(lh_run_means, axis=0))
        rest_subject_means.append(np.mean(rest_run_means, axis=0))

    return (
        np.asarray(rh_subject_means),
        np.asarray(lh_subject_means),
        np.asarray(rest_subject_means),
    )

rh_means, lh_means, rest_means = hand_and_rest_means_by_subject()
n_subjects = rh_means.shape[0]

# Paired/one-sample t-tests across subjects for each parcel
rh_minus_rest = rh_means - rest_means
lh_minus_rest = lh_means - rest_means
rh_minus_lh = rh_means - lh_means

rh_t, rh_p = stats.ttest_1samp(rh_minus_rest, popmean=0, axis=0)
lh_t, lh_p = stats.ttest_1samp(lh_minus_rest, popmean=0, axis=0)
contrast_t, contrast_p = stats.ttest_rel(rh_means, lh_means, axis=0)

alpha = 0.05
rh_sig, rh_q = benjamini_hochberg(rh_p, alpha=alpha)
lh_sig, lh_q = benjamini_hochberg(lh_p, alpha=alpha)
contrast_sig, contrast_q = benjamini_hochberg(contrast_p, alpha=alpha)

summary_mask = rh_sig | lh_sig | contrast_sig

q2_results = pd.DataFrame({
    'region': np.arange(N_PARCELS),
    'name': region_info['name'],
    'hemi': region_info['hemi'],
    'network': region_info['network'],
    'rh_modulated': rh_sig,
    'lh_modulated': lh_sig,
    'rh_vs_lh_significant': contrast_sig,
})

network_summary = (
    q2_results.assign(any_significant=summary_mask)
    .groupby('network')
    .agg(
        n_regions=('region', 'size'),
        any_significant=('any_significant', 'sum'),
        rh_modulated=('rh_modulated', 'sum'),
        lh_modulated=('lh_modulated', 'sum'),
        rh_vs_lh_significant=('rh_vs_lh_significant', 'sum'),
    )
)

network_summary = network_summary.reindex([n for n in network_order if n in network_summary.index])
network_labels_ordered = [network_label_map[net] for net in network_summary.index]

fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(network_summary))
width = 0.25

ax.bar(x - width, network_summary['rh_modulated'], width, label='Right Hand vs Rest', color='tab:blue', alpha=0.85)
ax.bar(x, network_summary['lh_modulated'], width, label='Left Hand vs Rest', color='tab:green', alpha=0.85)
ax.bar(x + width, network_summary['rh_vs_lh_significant'], width, label='Right vs Left Hand', color='tab:red', alpha=0.85)

ax.set_ylabel('Number of Significant Regions')
ax.set_title('Significant Modulations by Functional Network', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(network_labels_ordered, rotation=45, ha='right')
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Subjects tested: {n_subjects}")
print(f"Right-hand modulation (q < 0.05): {rh_sig.sum()} regions")
print(f"Left-hand modulation (q < 0.05): {lh_sig.sum()} regions")
print(f"Right-vs-left contrast (q < 0.05): {contrast_sig.sum()} regions")

# Q3

In [ ]:
# First level GLM: fit all six motor conditions for each subject, then average across runs.
all_betas_subject_no_hrf = compute_subject_condition_betas(hrf=None)

# Group level focus: isolate the right hand beta map for every subject and parcel.
rh_betas_subject_no_hrf = all_betas_subject_no_hrf[:, rh_idx, :]
betas_standard_no_hrf = rh_betas_subject_no_hrf.mean(axis=0)

# Bootstrap 95% confidence intervals for the subject-level mean beta in each parcel.
rng = np.random.default_rng(42)
bootstrap_indices = rng.integers(
    0,
    len(subjects),
    size=(N_BOOTSTRAP, len(subjects)),
)
beta_boots = rh_betas_subject_no_hrf[bootstrap_indices].mean(axis=1) # 2000 x 360
ci_low, ci_high = np.percentile(beta_boots, [2.5, 97.5], axis=0)

# Test whether each parcel's right-hand beta is significantly above zero across subjects.
t_glm_no_hrf, p_glm_no_hrf = stats.ttest_1samp(
    rh_betas_subject_no_hrf,
    popmean=0,
    axis=0,
    alternative='greater',
)
glm_sig_no_hrf, glm_q_no_hrf = benjamini_hochberg(p_glm_no_hrf, alpha=0.05)
n_sig_no_hrf = int(glm_sig_no_hrf.sum())
print(f"group level GLM found: {n_sig_no_hrf} / {N_PARCELS} parcels have significantly positive right hand task activation (FDR q < 0.05)")

# Q4

In [ ]:
hrf = spm_hrf(
    t_r=TR,
    oversampling=1,
    time_length=32.0,
    onset=0.0
)
plt.figure(figsize=(8, 4))
plt.plot(np.arange(len(hrf)) * TR, hrf, color='black', linewidth=2)
plt.axhline(0, color='gray', linewidth=1, linestyle='--')
plt.xlabel('Time after stimulus onset (seconds)')
plt.ylabel('HRF amplitude')
plt.title('Double-Gamma Hemodynamic Response Function')
plt.grid(alpha=0.3)
plt.show()

# First level HRF GLM: fit all six HRF-convolved MOTOR predictors for each subject.
all_betas_subject = compute_subject_condition_betas(hrf=hrf)
all_betas = all_betas_subject.mean(axis=0)

# Group-level: isolate the right hand HRF beta map for every subject and parcel.
rh_betas_subject_hrf = all_betas_subject[:, rh_idx, :]
betas_standard_hrf = rh_betas_subject_hrf.mean(axis=0)

# Test whether each parcel's right hand HRF beta is significantly above zero across subjects.
t_glm_hrf, p_glm_hrf = stats.ttest_1samp(
    rh_betas_subject_hrf,
    popmean=0,
    axis=0,
    alternative='greater',
)
glm_sig_hrf, glm_q_hrf = benjamini_hochberg(p_glm_hrf, alpha=0.05)

n_sig_hrf = int(glm_sig_hrf.sum())
print(f"Group level GLM (with double-gamma HRF) beta test found: {n_sig_hrf} / {N_PARCELS} parcels have significantly positive right hand task activation (FDR q < 0.05)")

# Q3 vs Q4 Comparison: 4-panel figure
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Q3 (No HRF) vs Q4 (Double-Gamma HRF) — Right Hand Condition',
             fontsize=14, fontweight='bold', y=1.01)

# Significance-change labels for colouring scatter dots
both_sig   =  glm_sig_no_hrf &  glm_sig_hrf
hrf_gained = ~glm_sig_no_hrf &  glm_sig_hrf   # became significant with HRF
hrf_lost   =  glm_sig_no_hrf & ~glm_sig_hrf   # lost significance with HRF
neither    = ~glm_sig_no_hrf & ~glm_sig_hrf

# Panel 1: t-statistic scatter
ax = axes[0, 0]
t_min = min(t_glm_no_hrf.min(), t_glm_hrf.min()) - 0.5
t_max = max(t_glm_no_hrf.max(), t_glm_hrf.max()) + 0.5
ax.scatter(t_glm_no_hrf[neither], t_glm_hrf[neither], alpha=0.5,
           color='lightgray', s=18, label='Neither sig.')
ax.scatter(t_glm_no_hrf[both_sig], t_glm_hrf[both_sig], alpha=0.7,
           color='royalblue', s=18, label='Both sig.')
ax.scatter(t_glm_no_hrf[hrf_gained], t_glm_hrf[hrf_gained], alpha=0.9,
           color='seagreen',  s=28, zorder=5, label='Gained with HRF')
ax.scatter(t_glm_no_hrf[hrf_lost], t_glm_hrf[hrf_lost], alpha=0.9,
           color='tomato', s=28, zorder=5, label='Lost with HRF')
ax.plot([t_min, t_max], [t_min, t_max], 'k--', lw=1.2, label='y = x')
ax.set_xlabel('t-statistic  (Q3: No HRF)', fontsize=11)
ax.set_ylabel('t-statistic  (Q4: Double-Gamma HRF)', fontsize=11)
ax.set_title('t-Statistics per Parcel', fontsize=12)
ax.legend(fontsize=8, markerscale=1.4)
ax.grid(True, alpha=0.25)

# Panel 2: overlapping t-statistic distributions
ax = axes[0, 1]
bins = np.linspace(t_min, t_max, 40)
ax.hist(t_glm_no_hrf, bins=bins, alpha=0.55, color='slategray',
        label='Q3 No HRF', edgecolor='white', linewidth=0.4)
ax.hist(t_glm_hrf,    bins=bins, alpha=0.55, color='darkorange',
        label='Q4 HRF',    edgecolor='white', linewidth=0.4)
ax.axvline(t_glm_no_hrf.mean(), color='slategray',  linestyle='--', lw=1.5,
           label=f'Mean Q3 = {t_glm_no_hrf.mean():.2f}')
ax.axvline(t_glm_hrf.mean(),    color='darkorange',  linestyle='--', lw=1.5,
           label=f'Mean Q4 = {t_glm_hrf.mean():.2f}')
ax.set_xlabel('t-statistic', fontsize=11)
ax.set_ylabel('Number of parcels', fontsize=11)
ax.set_title('Distribution of t-Statistics', fontsize=12)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.25)

# Panel 3: beta scatter
ax = axes[1, 0]
b_min = min(betas_standard_no_hrf.min(), betas_standard_hrf.min()) - 0.02
b_max = max(betas_standard_no_hrf.max(), betas_standard_hrf.max()) + 0.02
ax.scatter(betas_standard_no_hrf[neither],    betas_standard_hrf[neither],
           alpha=0.5, color='lightgray', s=18, label='Neither sig.')
ax.scatter(betas_standard_no_hrf[both_sig],   betas_standard_hrf[both_sig],
           alpha=0.7, color='royalblue', s=18, label='Both sig.')
ax.scatter(betas_standard_no_hrf[hrf_gained], betas_standard_hrf[hrf_gained],
           alpha=0.9, color='seagreen',  s=28, zorder=5, label='Gained with HRF')
ax.scatter(betas_standard_no_hrf[hrf_lost],   betas_standard_hrf[hrf_lost],
           alpha=0.9, color='tomato',    s=28, zorder=5, label='Lost with HRF')
ax.plot([b_min, b_max], [b_min, b_max], 'k--', lw=1.2, label='y = x')
ax.set_xlabel('Mean Beta  (Q3: No HRF)', fontsize=11)
ax.set_ylabel('Mean Beta  (Q4: Double-Gamma HRF)', fontsize=11)
ax.set_title('Beta Estimates per Parcel', fontsize=12)
ax.legend(fontsize=8, markerscale=1.4)
ax.grid(True, alpha=0.25)

# Panel 4: significant parcel counts per functional network
ax = axes[1, 1]
net_labels_plot = network_labels[:len(network_order)]
n_sig_no_hrf_net = np.array([
    glm_sig_no_hrf[np.array(region_info['network']) == net].sum()
    for net in network_order
])
n_sig_hrf_net = np.array([
    glm_sig_hrf[np.array(region_info['network']) == net].sum()
    for net in network_order
])
x_pos = np.arange(len(network_order))
bar_w = 0.38
ax.bar(x_pos - bar_w/2, n_sig_no_hrf_net, width=bar_w, color='slategray',
       alpha=0.75, label='Q3 No HRF', zorder=3)
ax.bar(x_pos + bar_w/2, n_sig_hrf_net,    width=bar_w, color='darkorange',
       alpha=0.75, label='Q4 HRF',    zorder=3)
ax.set_xticks(x_pos)
ax.set_xticklabels(net_labels_plot, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Significant parcels (FDR q < 0.05)', fontsize=10)
ax.set_title('Significant Parcels per Network', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, axis='y', alpha=0.25, zorder=0)

plt.tight_layout()
plt.show()

---
## Question 5: Representational Dissimilarity Matrices (RDMs)

**Goal**
Use the estimated beta maps to compare the spatial activation patterns for all six motor conditions and map how representational geometry varies across functional networks.

**Method**
Subject-level HRF GLM beta maps are estimated for each condition and averaged across subjects. The whole-cortex RDM treats each condition beta map as a 360-dimensional activation pattern and computes pairwise correlation distance. To map variation across the brain, the same RDM calculation is repeated separately inside each functional network.

**Interpretation**
Small distances indicate conditions with similar distributed activation patterns; large distances indicate more distinct patterns. Network-specific RDMs show whether different cortical systems carry similar or different motor-condition geometries. This supports representational analysis without claiming a complete model of motor cortex.



In [ ]:
all_betas = all_betas_subject.mean(axis=0)  # conditions x parcels
motor_conditions = conditions

# Whole-cortex RDM: correlation distance between condition-level beta patterns.
distances = pdist(all_betas, metric='correlation')
brain_rdm = squareform(distances)

plt.figure(figsize=(7, 6))
sns.heatmap(brain_rdm, xticklabels=motor_conditions, yticklabels=motor_conditions, cmap='viridis', annot=True, square=True)
plt.title('Whole-Cortex Brain RDM (Correlation Distance)')
plt.tight_layout()
plt.show()

# Map representational variation across the brain by computing one RDM per network.
network_rdms = {}
network_rdm_rows = []
upper_idx = np.triu_indices(len(motor_conditions), k=1)

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()
for ax, net, label in zip(axes, network_order, network_labels):
    idx = np.where(region_info['network'] == net)[0]
    net_betas = all_betas[:, idx]
    net_rdm = squareform(pdist(net_betas, metric='correlation'))
    network_rdms[net] = net_rdm

    corr_to_whole, p_to_whole = stats.spearmanr(brain_rdm[upper_idx], net_rdm[upper_idx])
    network_rdm_rows.append({
        'network': net,
        'label': label,
        'n_regions': len(idx),
        'rdm_similarity_to_whole_brain': corr_to_whole,
        'p_uncorrected': p_to_whole,
    })

    sns.heatmap(net_rdm, ax=ax, xticklabels=motor_conditions, yticklabels=motor_conditions, cmap='viridis', cbar=False, square=True)
    ax.set_title(label, fontsize=10)

for ax in axes[len(network_order):]:
    ax.axis('off')

fig.suptitle('Network-Specific RDMs: Representational Geometry across Cortex', fontsize=16)
plt.tight_layout()
plt.show()

network_rdm_summary = pd.DataFrame(network_rdm_rows).sort_values('rdm_similarity_to_whole_brain', ascending=False)
display(network_rdm_summary)

---
## Question 6: Linear Decoding of Motor Stimulus Type

**Goal**
Test whether multivariate cortical activity patterns contain enough information to predict the MOTOR stimulus condition: `lf`, `rf`, `lh`, `rh`, `t`, or `cue` using a purely linear classifier.

**Method**
Each sample is a subject-run condition mean. For every subject and run, the 360-parcel mean activity pattern is computed separately for all six MOTOR conditions. The train/test split is grouped by subject, so the same subject cannot appear in both training and test sets. A multinomial logistic-regression decoder is trained on standardized parcel features. Principal Component Analysis (PCA) variants test how decoding changes as the feature space is compressed, satisfying the template option of whole-brain or dimensionality-reduced decoding.

**Interpretation**
Above-chance six-way accuracy suggests that stimulus identity is separable in distributed cortical activity. We strictly use Logistic Regression as the linear decoder as requested, avoiding deep non-linear models that could overfit the relatively small sample size.


In [ ]:
# --- QUESTION 6: Linear Decoder ---
# Subject-level six-way decoder for MOTOR stimulus type using Logistic Regression
print("Building Linear Decoder (Logistic Regression)...")
X_decoder = []
Y_decoder = []
groups = []

for subj_i in range(len(subjects)):
    for run in range(len(RUNS)):
        ts = all_ts[run][subj_i]
        run_evs = all_evs[run][subj_i]
        for cond_idx, cond in enumerate(conditions):
            frames = condition_frames(run_evs, cond_idx)
            X_decoder.append(ts[:, frames].mean(axis=1))
            Y_decoder.append(cond_idx)
            groups.append(subj_i)

X_decoder = np.asarray(X_decoder)
Y_decoder = np.asarray(Y_decoder)
groups = np.asarray(groups)

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X_decoder, Y_decoder, groups=groups))
X_train, X_test = X_decoder[train_idx], X_decoder[test_idx]
y_train, y_test = Y_decoder[train_idx], Y_decoder[test_idx]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

linear_decoder = LogisticRegression(max_iter=2000, random_state=42)
linear_decoder.fit(X_train_scaled, y_train)
y_pred = linear_decoder.predict(X_test_scaled)
acc_whole = accuracy_score(y_test, y_pred)
chance = 1 / len(conditions)

print(f"Six-way Logistic Regression (360 regions): {acc_whole * 100:.2f}%")
print(f"Chance level: {chance * 100:.2f}%")

max_pca = min(X_train_scaled.shape[0], X_train_scaled.shape[1])
n_components_list = [n for n in [2, 5, 10, 20, 50, 100, 200, 360] if n <= max_pca]
print(f'Evaluating PCA range: up to {max_pca} components')

pca_full = PCA(n_components=max_pca, random_state=42)
X_train_pca_full = pca_full.fit_transform(X_train_scaled)
X_test_pca_full = pca_full.transform(X_test_scaled)

pca_accuracies = []
for n_pc in n_components_list:
    X_train_pca = X_train_pca_full[:, :n_pc]
    X_test_pca = X_test_pca_full[:, :n_pc]
    clf = LogisticRegression(max_iter=2000, random_state=42)
    clf.fit(X_train_pca, y_train)
    pca_accuracies.append(accuracy_score(y_test, clf.predict(X_test_pca)))
    print(f"  PCA ({n_pc:3d} components): {pca_accuracies[-1] * 100:.2f}%")

# Plot Results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(n_components_list, [a * 100 for a in pca_accuracies], marker='o', color='steelblue', linewidth=2, markersize=7)
axes[0].axhline(acc_whole * 100, color='tomato', linestyle='--', linewidth=1.5, label=f'Whole-brain baseline ({acc_whole * 100:.1f}%)')
axes[0].axhline(chance * 100, color='black', linestyle=':', linewidth=1.2, label='Chance')
axes[0].set_xlabel('Number of PCA Components')
axes[0].set_ylabel('Decoding Accuracy (%)')
axes[0].set_title('PCA Dimensionality Reduction vs Decoding Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.4)
axes[0].set_xscale('log')

cm = confusion_matrix(y_test, y_pred, labels=np.arange(len(conditions)))
sns.heatmap(cm, ax=axes[1], annot=True, fmt='d', cmap='Blues', xticklabels=conditions, yticklabels=conditions, cbar=False)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
axes[1].set_title('Six-Way Logistic Regression Confusion Matrix')

plt.tight_layout()
plt.show()

---
## Question 7: Compare Brain Representations to a Template

**Goal**
Test whether the empirical brain RDM matches a simple theoretical model in which hands are similar to hands, feet are similar to feet, and other body-part comparisons are more distinct.

**Method**
A binary template RDM encodes the hypothesized body-part grouping. Spearman correlation compares the upper triangle of the empirical RDM with the template RDM.

**Interpretation**
A positive correlation would support the idea that the distributed activation patterns contain body-part structure. This is a deliberately simple model: it tests one interpretable hypothesis, but it does not capture graded somatotopy, hemispheric lateralization, or the full geometry of motor cortex.



In [ ]:
# 1. Create a Theoretical Template RDM
# Hands are similar to each other (dist=0), Feet are similar (dist=0), else (dist=1).
template_rdm = np.array([
    [0, 0, 1, 1, 1, 1], # lf
    [0, 0, 1, 1, 1, 1], # rf
    [1, 1, 0, 0, 1, 1], # lh
    [1, 1, 0, 0, 1, 1], # rh
    [1, 1, 1, 1, 0, 1], # t
    [1, 1, 1, 1, 1, 0]  # cue
])

correlation, p_val = stats.spearmanr(brain_rdm[upper_idx], template_rdm[upper_idx])
print(f"Spearman Correlation (Brain vs Template): {correlation:.3f} (p={p_val:.3f})")

---
## Question 8: Artificial Neural Network (ANN) Comparison

**Goal**
Contrast the representational geometry of the human brain to an artificial neural network (ANN) designed to solve a similar task.

**Method**
We build a PyTorch Multi-Class ANN with a single hidden layer. We compare the Representational Dissimilarity Matrix (RDM) of the human brain to the RDM of the ANN's hidden layer before training (untrained) and after training on the brain data to classify the 6 motor conditions.

**Interpretation: Why does the Untrained ANN have a higher correlation with the brain than the Trained ANN?**

This is a mathematically fascinating concept in Representational Similarity Analysis (RSA):

**1. The Untrained ANN is just a "Random Projection" of the Brain**
In the code, you are feeding the actual brain activity patterns (the GLM betas) directly into the ANN. When a neural network is untrained, its weights are completely random. Mathematically, multiplying the brain data by a random weight matrix acts as a random projection. According to the Johnson-Lindenstrauss lemma, random projections preserve the relative distances and geometry of the input data.
* **Result:** Because the input is the brain data, the untrained hidden layer simply reflects the original geometry of the brain data. Therefore, the Untrained RDM looks almost identical to the Human Brain RDM (Correlation = 0.90).

**2. The Trained ANN "Destroys" the Brain's Geometry**
When you train the network, the loss function (`CrossEntropyLoss`) tells the network to classify the 6 conditions into 6 distinct, independent categories. To get 100% accuracy, the network learns to orthogonalize the representations.
* It forces the hidden representations of the 6 conditions to be as far apart and equidistant from each other as possible.
* It completely ignores the fact that in the real human brain, the "left foot" and "right foot" have highly overlapping, similar representations (because their physical locations on the motor homunculus are right next to each other).
* **Result:** The trained network reshapes the hidden layer into a rigid, categorical geometry that no longer resembles the natural, overlapping biological similarities of the human brain. Thus, the correlation drops to 0.59.

**Summary**
The brain has a naturally "messy" representation where similar body parts overlap. The untrained network preserves this messiness because it hasn't learned to categorize anything yet. The trained network becomes too good at the task—it warps the representations to be perfectly distinct, which makes it an excellent classifier, but less "brain-like"!

*(Note: In real computational neuroscience research, instead of training the ANN on the brain data, researchers train the ANN on a separate task like processing images or raw motor kinematics, and then compare the fully trained ANN's hidden layers to the brain to see if they naturally arrived at the same geometry.)*


In [ ]:
# 1. Build a PyTorch Multi-Class ANN
class MotorClassifierANN(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(360, 128) # Hidden layer (This is the representation we want!)
        self.relu = nn.ReLU()
        self.output = nn.Linear(128, 6)   # 6 output classes (lf, rf, lh, rh, t, cue)
        
    def forward(self, x):
        h = self.relu(self.hidden(x))
        out = self.output(h)
        return out, h # Return both the prediction and the hidden layer!

# 2. Extract Untrained ANN RDM (Random Initialization geometry)
ann_multi = MotorClassifierANN()
ann_multi.eval()

# We pass the average brain activity (the Betas from Q5) into the network to see its hidden representation
with torch.no_grad():
    input_tensor = torch.FloatTensor(all_betas)
    _, untrained_hidden = ann_multi(input_tensor)
    
untrained_ann_rdm = squareform(pdist(untrained_hidden.numpy(), metric='correlation'))

# 3. Quick "Training" simulation (force the hidden layer to learn the conditions)
# In real research, we'd train this on thousands of trials. Here we do a fast overfit on the averages for demonstration.
optimizer = optim.Adam(ann_multi.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
target = torch.LongTensor([0, 1, 2, 3, 4, 5]) # The 6 labels

ann_multi.train()
for _ in range(50):
    optimizer.zero_grad()
    preds, _ = ann_multi(input_tensor)
    loss = criterion(preds, target)
    loss.backward()
    optimizer.step()

# 4. Extract Trained ANN RDM
ann_multi.eval()
with torch.no_grad():
    _, trained_hidden = ann_multi(input_tensor)
    
trained_ann_rdm = squareform(pdist(trained_hidden.numpy(), metric='correlation'))

# 5. Statistical Contrast: Human Brain RDM vs. Untrained ANN vs. Trained ANN
corr_untrained, _ = stats.spearmanr(brain_rdm[upper_idx], untrained_ann_rdm[upper_idx])
corr_trained, _ = stats.spearmanr(brain_rdm[upper_idx], trained_ann_rdm[upper_idx])

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.heatmap(brain_rdm, ax=axes[0], xticklabels=motor_conditions, yticklabels=motor_conditions, cmap='viridis', cbar=False)
axes[0].set_title("Human Brain RDM")

sns.heatmap(untrained_ann_rdm, ax=axes[1], xticklabels=motor_conditions, yticklabels=motor_conditions, cmap='gray', cbar=False)
axes[1].set_title('Untrained PyTorch ANN\nCorr: {:.2f}'.format(corr_untrained))

sns.heatmap(trained_ann_rdm, ax=axes[2], xticklabels=motor_conditions, yticklabels=motor_conditions, cmap='plasma', cbar=False)
axes[2].set_title('Trained PyTorch ANN\nCorr: {:.2f}'.format(corr_trained))

plt.tight_layout()
plt.show()